# Consumir un Claude Managed Agent vía *sessions*

En la lección 2 desplegamos nosotros la API (FastAPI + Docker + fly.io). En esta lección **Anthropic hospeda el agente completo** y nosotros solo lo consumimos a través de **sesiones**: un log de eventos durable que vive en el servidor.

El agente es el **"Analista Económico Chile"** (definido en `agent.py`, en esta misma carpeta):
- **Ejecuta código Python** en un sandbox que solo puede alcanzar [mindicador.cl](https://mindicador.cl/api) → datos duros (UF, dólar, UTM, IPC…).
- Tiene **web_search / web_fetch** habilitados → contexto más amplio (noticias, causas).
- Escribe informes en `/mnt/session/outputs/` → los descargamos por la Files API.

**Requisito:** un `ANTHROPIC_API_KEY` con acceso al beta de Managed Agents en tu `.env` (copia `.env.example`).

> ⚠️ Cada mensaje consume tokens reales — mantén las preguntas cortas y borra la sesión al final.

## 1 · Setup: aprovisionar el agente (idempotente)

Un agente son **3 llamadas de API**: `environments.create` → `agents.create` → `sessions.create`. Las dos primeras se hacen **una sola vez** — `provision()` cachea los ids en `agent_config.json` y las corridas siguientes los reutilizan.

In [ ]:
# `agent.py` vive en esta misma carpeta. Aseguramos que el directorio del
# notebook esté en sys.path (Jupyter usa esta carpeta como cwd) e importamos.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import agent

cfg = agent.provision()
cfg

## 2 · Primera sesión: datos duros desde el sandbox

Creamos una sesión (queda **persistida en el servidor**, no en este notebook) y preguntamos por la UF. Fíjate en las líneas `⚙️ [sandbox · ...]`: es el agente **ejecutando Python** contra mindicador.cl — nunca le escribimos una tool `get_uf`, solo le describimos la API en el system prompt.

In [ ]:
session_id = agent.start_session(cfg["agent_id"], cfg["environment_id"], title="Demo clase — 20260708")
print(f"Sesión creada: {session_id}\n")

_ = agent.print_reply(session_id, "¿Cuál es el valor de la UF hoy? ¿Cuántos pesos son 500 UF?")

## 3 · Análisis con contexto: código + web + informe descargable

Ahora un encargo más grande, que combina las dos capacidades: la **serie histórica** (código en el sandbox) y el **por qué** (web_search). Le pedimos además que deje un informe en `/mnt/session/outputs/` para descargarlo.

In [ ]:
_ = agent.print_reply(
    session_id,
    "Baja la serie del dólar observado del último año desde mindicador.cl, "
    "calcula estadísticas clave (mín, máx, promedio, variación %), y busca en la web "
    "qué explica los movimientos recientes. Escribe un informe breve en markdown "
    "en /mnt/session/outputs/informe.md y dame un resumen de 3 líneas aquí.",
)

In [ ]:
# Descargamos lo que el agente escribió en /mnt/session/outputs/ → ./outputs/
archivos = agent.download_outputs(session_id)
archivos

In [ ]:
from IPython.display import Markdown

informe = next((a for a in archivos if a.suffix == ".md"), None)
Markdown(informe.read_text()) if informe else archivos

## 4 · Sesiones durables: el estado vive en el servidor (la gracia)

En la lección 2 el estado moría con cada request. Acá la sesión es un **log de eventos append-only** en el servidor: podemos listar las sesiones del agente, **reproducir** la conversación completa desde el log, y **reanudarla** — incluso si reinicias el kernel de este notebook o te cambias de computador.

*(Prueba real: reinicia el kernel, corre solo la celda de setup (§1) y luego estas — la conversación sigue ahí.)*

In [ ]:
# Todas las sesiones de este agente, persistidas en el servidor
for s in agent.list_sessions(cfg["agent_id"]):
    print(f"{s.created_at:%Y-%m-%d %H:%M}  ·  {s.status:<10}  ·  {s.id}")

In [ ]:
# Reproducir la conversación desde el log de eventos (no guardamos nada localmente)
session_id = agent.list_sessions(cfg["agent_id"])[0].id  # la más reciente

for rol, texto in agent.load_history(session_id):
    if rol == "tool":
        print(f"   ⚙️  {texto}")
    else:
        print(f"[{rol}] {texto[:200]}{'…' if len(texto) > 200 else ''}\n")

In [ ]:
# Reanudar la MISMA sesión con una pregunta de seguimiento:
# el agente recuerda todo el contexto anterior (UF, dólar) sin que se lo repitamos.
_ = agent.print_reply(session_id, "¿Y cómo se compara eso con la UTM de este mes?")

## 5 · Limpieza (opcional)

Borramos la sesión de demo. El agente y el ambiente son solo definiciones — no cuestan nada mantenerlos.

In [ ]:
# agent.delete_session(session_id)
# print("Sesión eliminada")

## 6 · Tracing con LangSmith → correr evals

Hasta aquí *usamos* el agente. Ahora lo **instrumentamos** para poder evaluarlo.

El bucle del agente (el "cerebro" + el sandbox) corre en los servidores de Anthropic: desde el cliente solo vemos la sesión — el mensaje que enviamos y los eventos (`agent.message`, `agent.tool_use`) que vuelven. Por eso no sirve un auto-instrumentador de llamadas `messages.create`; usamos el decorador **`@traceable`** de LangSmith sobre la función de turno del cliente (`agent.ask_agent`). Cada turno se convierte en un *run* de LangSmith que luego alimenta `evaluate(...)`, igual que en la Clase 3.6 · Lección 1.

**Requisito:** un `LANGSMITH_API_KEY` en tu `.env` (copia `.env.example`).

In [ ]:
# --- Setup LangSmith: tracing on + credenciales ---
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langsmith==0.9.5", "python-dotenv"])

os.environ.setdefault("LANGSMITH_PROJECT", os.getenv("LANGSMITH_PROJECT", "analista-economico-cl"))
os.environ.setdefault("LANGSMITH_TRACING", "true")

if IN_COLAB:
    from google.colab import userdata
    for k in ("ANTHROPIC_API_KEY", "LANGSMITH_API_KEY"):
        val = userdata.get(k)
        if val:
            os.environ[k] = val
else:
    from dotenv import load_dotenv
    for p in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
        if p.exists():
            load_dotenv(p)
            break

if not os.getenv("LANGSMITH_API_KEY"):
    raise EnvironmentError("Falta LANGSMITH_API_KEY — agrégala a tu .env (ver .env.example).")

# Recarga agent para que tome langsmith.traceable (necesario en Colab tras instalar).
import importlib
import agent
importlib.reload(agent)

print("LangSmith tracing ON · proyecto:", os.environ["LANGSMITH_PROJECT"])

### Un turno traceado

`agent.ask_agent(pregunta)` abre una **sesión nueva**, hace la pregunta y devuelve `{"output", "tool_calls", "session_id"}`. Está decorada con `@traceable`, así que cada llamada queda registrada como un run en tu proyecto de LangSmith — con la pregunta como input, la respuesta como output y los eventos del sandbox en la metadata del run.

> ¿Por qué una sesión nueva por turno? Para evaluar necesitamos turnos **independientes**: que un ejemplo no herede el contexto de otro. (`print_reply`, arriba, es lo contrario: conversa sobre una misma sesión.)

In [ ]:
# Un turno traceado en su propia sesión. Aparecerá como un run en LangSmith.
salida = agent.ask_agent("¿Cuál es el valor de la UF hoy? ¿Cuántos pesos son 500 UF?")
print(salida["output"])
print("\nTool calls en el sandbox:", salida["tool_calls"])

### Un dataset pequeño para evaluar

Estos ejemplos no llevan respuesta de referencia (los indicadores cambian a diario): evaluaremos **propiedades** de la respuesta, no una cifra exacta.

In [ ]:
from langsmith import Client

ls_client = Client()
DATASET = "analista-economico-cl — demo"

ejemplos = [
    {"inputs": {"question": "¿Cuál es el valor de la UF hoy?"}},
    {"inputs": {"question": "¿A cuántos pesos equivalen 100 dólares observados hoy?"}},
    {"inputs": {"question": "¿Cuál es la UTM de este mes?"}},
]

if ls_client.has_dataset(dataset_name=DATASET):
    ds = ls_client.read_dataset(dataset_name=DATASET)
    print(f"Usando dataset existente: {ds.name}")
else:
    ds = ls_client.create_dataset(DATASET, description="Indicadores económicos de Chile")
    ls_client.create_examples(dataset_id=ds.id, examples=ejemplos)
    print(f"Dataset '{ds.name}' creado con {len(ejemplos)} ejemplos")

dataset_name = DATASET

### Evaluadores

Tres evaluadores, del más barato al más caro:
- `tiene_cifra` — determinista: la respuesta contiene al menos un número.
- `uso_sandbox` — determinista: el agente ejecutó Python en el sandbox (lee `tool_calls` de la salida del run).
- `cita_datos` — LLM-as-judge (Anthropic): la respuesta da una cifra concreta **y** cita `mindicador.cl`.

In [ ]:
import json
import re

from langsmith.schemas import Example, Run

JUDGE_MODEL = "claude-sonnet-5"  # juez LLM (mismo cliente Anthropic del agente)


def _judge(question: str, answer: str, criterio: str) -> dict:
    """LLM-as-judge con Anthropic + structured outputs."""
    resp = agent.client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=400,
        system="Eres un evaluador de respuestas de un analista económico de Chile. Sé estricto y objetivo.",
        messages=[{"role": "user", "content": (
            f"Criterio: {criterio}\n\n"
            f"Pregunta: {question}\n\n"
            f"Respuesta del agente:\n{answer}\n\n"
            "¿La respuesta cumple el criterio?"
        )}],
        output_config={"format": {"type": "json_schema", "schema": {
            "type": "object",
            "properties": {
                "cumple": {"type": "boolean"},
                "motivo": {"type": "string"},
            },
            "required": ["cumple", "motivo"],
            "additionalProperties": False,
        }}},
    )
    texto = next(b.text for b in resp.content if getattr(b, "type", None) == "text")
    return json.loads(texto)


# Determinista: ¿la respuesta trae al menos una cifra?
def tiene_cifra(run: Run, example: Example) -> dict:
    ans = (run.outputs or {}).get("output", "")
    return {"key": "tiene_cifra", "score": 1.0 if re.search(r"\d", ans) else 0.0}


# Determinista: ¿el agente realmente ejecutó el sandbox?
def uso_sandbox(run: Run, example: Example) -> dict:
    calls = (run.outputs or {}).get("tool_calls", [])
    return {"key": "uso_sandbox", "score": 1.0 if calls else 0.0,
            "comment": f"{len(calls)} tool call(s): {calls}"}


# LLM-as-judge: ¿da una cifra concreta y cita mindicador.cl?
def cita_datos(run: Run, example: Example) -> dict:
    ans = (run.outputs or {}).get("output", "")
    v = _judge(example.inputs["question"], ans,
               "La respuesta entrega una cifra concreta y cita mindicador.cl como fuente.")
    return {"key": "cita_datos", "score": 1.0 if v["cumple"] else 0.0, "comment": v["motivo"]}

In [ ]:
# ⚠️ Cada ejemplo abre una sesión nueva y consume tokens reales. Mantén el dataset chico.
from langsmith import evaluate


def agent_target(inputs: dict) -> dict:
    # target de langsmith: recibe los inputs del ejemplo y devuelve el dict de ask_agent
    return agent.ask_agent(inputs["question"])


results = evaluate(
    agent_target,
    data=dataset_name,
    evaluators=[tiene_cifra, uso_sandbox, cita_datos],
    experiment_prefix="analista-economico-cl",
    max_concurrency=2,
)
print("Experimento:", results.experiment_name)

### Inspeccionar y evaluar en LangSmith

- Cada llamada a `agent.ask_agent(...)` y cada ejemplo del experimento queda como un **run** en tu proyecto `LANGSMITH_PROJECT`.
- Abre [smith.langchain.com](https://smith.langchain.com) → tu proyecto para ver la pregunta, la respuesta, los `tool_calls` del sandbox (en la metadata del run) y los scores de los evaluadores.
- Desde ahí puedes **crear datasets a partir de runs reales**, comparar experimentos entre versiones del `system` prompt del agente, o sumar evaluadores.

> Reutiliza los patrones de `class_3_6_production/leccion1_evaluacion_langsmith` para evaluadores LLM-as-judge más elaborados.